In [1]:
import numpy as np
import pandas as pd

from gplearn.genetic import SymbolicRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler


In [2]:
# ============================================================
# LOAD DATA
# ============================================================

data = np.load(
    "../../data/c_a/processed/dataset_full.npz",
    allow_pickle=True
)

X_full = data["X"]

df = pd.DataFrame(
    X_full,
    columns=[f"F{i}" for i in range(30)]
)

print("Original shape:", df.shape)

Original shape: (6205696, 30)


In [3]:
# ============================================================
# OPTIONAL SUBSAMPLING
# ============================================================

MAX_SAMPLES = 2500

if len(df) > MAX_SAMPLES:
    df = df.sample(
        n=MAX_SAMPLES,
        random_state=42
    )

print("Using shape:", df.shape)



Using shape: (2500, 30)


In [4]:

# ============================================================
# SCALE DATA
# ============================================================

# Uncomment if wanted:
# X_scaled = scaler.fit_transform(df)

X_scaled = df.values.astype(np.float64)

In [5]:

indices = np.arange(len(X_scaled))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
)


In [7]:



# ============================================================
# RESULTS STORAGE
# ============================================================

results = []

TARGET_FEATURES = [3, 5, 16, 18]
# ============================================================
# LOOP OVER ALL FEATURES
# ============================================================

for TARGET_FEATURE in TARGET_FEATURES:

    print("\n" + "=" * 80)
    print(f"TARGET FEATURE: F{TARGET_FEATURE}")
    print("=" * 80)

    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------

    y = X_scaled[:, TARGET_FEATURE]

    # --------------------------------------------------------
    # INPUT FEATURES
    # --------------------------------------------------------

    X_other = np.delete(
        X_scaled,
        TARGET_FEATURE,
        axis=1
    )

    # --------------------------------------------------------
    # FEATURE MAPPING
    # --------------------------------------------------------

    remaining_features = [
        i for i in range(X_scaled.shape[1])
        if i != TARGET_FEATURE
    ]

    print("\nFeature mapping:")
    # print(f"X{local_idx}")
    # for local_idx, original_idx in enumerate(remaining_features):
    #     print(f"x{local_idx} -> F{original_idx}")

    # --------------------------------------------------------
    # TRAIN / TEST SPLIT
    # --------------------------------------------------------

    X_train = X_other[train_idx]
    X_test  = X_other[test_idx]
    
    y_train = y[train_idx]
    y_test  = y[test_idx]
    # ========================================================
    # GP MODEL
    # ========================================================

    model = SymbolicRegressor(

        # ----------------------------------------------------
        # EVOLUTION
        # ----------------------------------------------------

        population_size=2500,
        generations=80,

        # ----------------------------------------------------
        # TOURNAMENT
        # ----------------------------------------------------

        tournament_size=20,

        # ----------------------------------------------------
        # STOPPING
        # ----------------------------------------------------

        stopping_criteria=0.000001,

        # ----------------------------------------------------
        # MUTATION / CROSSOVER
        # ----------------------------------------------------

        p_crossover=0.75,
        p_subtree_mutation=0.1,
        p_hoist_mutation=0.05,
        p_point_mutation=0.1,

        # ----------------------------------------------------
        # COMPLEXITY CONTROL
        # ----------------------------------------------------

        parsimony_coefficient=1e-4,

        init_depth=(2, 6),

        # ----------------------------------------------------
        # FUNCTIONS
        # ----------------------------------------------------


        function_set = (
            "add",   # addition
            "sub",   # subtraction
            "mul",   # multiplication
            "div",   # protected division
        
            "sqrt",  # protected sqrt
            "log",   # protected log
            "abs",   # absolute value
            "neg",   # negation
            "inv",   # protected inverse
        
            "max",   # maximum
            "min",   # minimum
        
            "sin",   # sine
            "cos",   # cosine
            "tan",   # tangent
        ),



        # ----------------------------------------------------
        # SAMPLING
        # ----------------------------------------------------

        max_samples=0.9,

        # ----------------------------------------------------
        # REPRODUCIBILITY
        # ----------------------------------------------------

        random_state=42,

        # ----------------------------------------------------
        # MULTIPROCESSING
        # ----------------------------------------------------

        n_jobs=-1,

        # ----------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------

        verbose=1,
    )

    # ========================================================
    # FIT
    # ========================================================

    model.fit(X_train, y_train)

    # ========================================================
    # PREDICT
    # ========================================================

    pred = model.predict(X_test)

    r2 = r2_score(y_test, pred)

    # ========================================================
    # EQUATION
    # ========================================================

    equation = str(model._program)

    print("\nFINAL EQUATION") 
    print(f"F{TARGET_FEATURE} = {equation}") 
    print("Fitness:", model._program.fitness_) 
    print("Length:", model._program.length_) 
    print("Depth:", model._program.depth_)
    
    # ========================================================
    # COMPLEXITY
    # ========================================================

    complexity = model._program.length_

    # ========================================================
    # FILTER STRONG EQUATIONS
    # ========================================================

    if r2 < 0.99:
        continue

    # ========================================================
    # STORE RESULTS
    # ========================================================

    results.append({
        "target_feature": TARGET_FEATURE,
        "r2": r2,
        "complexity": complexity,
        "equation": equation,
    })

    # ========================================================
    # PRINT
    # ========================================================

    print("\nRESULTS")
    print("-" * 40)

    print("R²:", r2)
    print("Complexity:", complexity)

    print("\nEquation:")
    print(f"F{TARGET_FEATURE} =", equation)






TARGET FEATURE: F3

Feature mapping:
    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0     8.24        1.313e+10        4          1201.35          1325.63      3.67m
   1     5.49      1.23863e+09        8          1119.18          1219.76      2.36m
   2     4.84        6.215e+06        7          1117.07          1238.71      2.73m
   3     4.53      1.12726e+07       11          1118.93          1222.02      2.98m
   4     5.55      1.62176e+06        8          1110.64          1296.62      2.96m
   5     7.36      7.48147e+08        8          1108.47          1316.13      2.95m
   6     8.33      5.53228e+06       12          993.529           899.04      2.73m
   7     9.69           958289       12          977.002          1047.79      3.20m
   8    11.27       7.3087e+06       

In [ ]:
print(model._program)

In [ ]:
results